# CATbot — Train and Export in Google Colab

This notebook trains the project's word-level Markov model from your own transcript `.txt` files and downloads a portable `catbot_model.json` for the Windows app. **No GPU is required.**

## Before running
1. Create `MyDrive/copycat-bot/raw` in Google Drive.
2. Upload several plain-text `.txt` transcripts from the **same speaker** into that folder.
3. Run the cells in order. Use transcripts you have permission to use. Aim for 20,000+ words if possible.


In [ ]:
from google.colab import drive, files
from pathlib import Path
import json, re, html, unicodedata, random
from collections import Counter, defaultdict

drive.mount('/content/drive')
DRIVE_FOLDER = Path('/content/drive/MyDrive/copycat-bot/raw')
transcript_files = sorted(DRIVE_FOLDER.glob('*.txt'))
if not transcript_files:
    raise FileNotFoundError(f'No .txt transcripts found in {DRIVE_FOLDER}. Upload transcript files to that Drive folder.')
print(f'Found {len(transcript_files)} transcript file(s):')
for f in transcript_files: print('-', f.name)


In [ ]:
# Clean and tokenize the transcripts (same overall cleaning approach as clean.py)
STAGE_WORDS = r'applause|laughter|laughs|cheers|cheering|music|audience|interruption|inaudible|pause|crosstalk|slogans|video|chanting|clapping|translation'
RE_TAGS = re.compile(r'<[^>]+>')
RE_URLS = re.compile(r'https?://\S+|www\.\S+')
RE_SQUARE = re.compile(r'\[[^\]]{0,100}\]')
RE_PAREN_STAGE = re.compile(rf'\((?=[^)]*(?:{STAGE_WORDS}))[^)]{{0,100}}\)', re.I)
TOKEN_RE = re.compile(r"[A-Za-z0-9]+(?:['-][A-Za-z0-9]+)*|[.!?,;:]")
MONTHS = r'(?:January|February|March|April|May|June|July|August|September|October|November|December|Jan|Feb|Mar|Apr|Jun|Jul|Aug|Sep|Sept|Oct|Nov|Dec)'
RE_DATES = re.compile(rf'\b\d{{1,2}}(?:st|nd|rd|th)?\s+{MONTHS}\.?,?\s+\d{{4}}\b|\b{MONTHS}\.?\s+\d{{1,2}}(?:st|nd|rd|th)?,?\s+\d{{4}}\b|\b\d{{1,2}}[/-]\d{{1,2}}[/-]\d{{2,4}}\b|\b\d{{4}}-\d{{2}}-\d{{2}}\b', re.I)
QUOTES = {'\u2018': "'", '\u2019': "'", '\u201c': '"', '\u201d': '"', '\u2013': '-', '\u2014': ' - ', '\u2026': '...'}

def clean_text(text):
    text = html.unescape(text)
    text = RE_TAGS.sub(' ', text)
    text = unicodedata.normalize('NFKC', text)
    for bad, good in QUOTES.items(): text = text.replace(bad, good)
    text = RE_URLS.sub(' ', text)
    text = RE_SQUARE.sub(' ', text)
    text = RE_PAREN_STAGE.sub(' ', text)
    text = RE_DATES.sub(' ', text)
    text = ''.join(ch for ch in text if ch.isprintable() or ch in '\n\t')
    return re.sub(r'\s+', ' ', text).strip()

docs = []
word_count = 0
for path in transcript_files:
    raw = path.read_text(encoding='utf-8', errors='replace')
    cleaned = clean_text(raw)
    tokens = TOKEN_RE.findall(cleaned)
    if len(tokens) > 2: docs.append(tokens)
    word_count += len(cleaned.split())
if not docs: raise ValueError('The transcript files did not contain enough usable text.')
print(f'Cleaned {len(docs)} document(s) | {word_count:,} words | {sum(map(len, docs)):,} tokens')
if word_count < 20000: print('WARNING: small dataset; generated text may repeat the training transcripts.')


In [ ]:
# Train the order-2 Markov chain
ORDER = 2  # model remembers the previous two tokens
SENTENCE_END = {'.', '!', '?'}
transitions = defaultdict(Counter)
starts = []
for tokens in docs:
    for i in range(len(tokens) - ORDER):
        state = tuple(t.lower() for t in tokens[i:i + ORDER])
        transitions[state][tokens[i + ORDER]] += 1
        if (i == 0 or tokens[i - 1] in SENTENCE_END) and state[0] not in SENTENCE_END | {',', ';', ':'}:
            starts.append(state)
print(f'Training complete: {len(transitions):,} states and {len(starts):,} sentence starts.')


In [ ]:
# Export the model for the Windows laptop
export_payload = {
    'format_version': 1,
    'model_type': 'word-level Markov chain',
    'order': ORDER,
    'source_files': [p.name for p in transcript_files],
    'word_count': word_count,
    'token_count': sum(len(d) for d in docs),
    'transitions': {'\u241f'.join(state): dict(counts) for state, counts in transitions.items()},
    'starts': [list(s) for s in starts]
}
output_path = Path('/content/catbot_model.json')
output_path.write_text(json.dumps(export_payload, ensure_ascii=False), encoding='utf-8')
print(f'Model saved: {output_path} ({output_path.stat().st_size / 1024:.1f} KB)')
files.download(str(output_path))


## Next step

Move the downloaded `catbot_model.json` into the `model` folder of the extracted `CATbot_Complete` project on your Windows laptop. Then run `python -m streamlit run app.py` from that project folder.
